In [1]:
#Parquet 읽고 쓰기
!pip install pandas pyarrow fastparquet -q

In [2]:
import pandas as pd
df = pd.DataFrame({"id":[1,2,3], "score":[0.8,0.6,0.9], "label":[1,0,1]})
df.to_parquet("example.parquet")   
print(pd.read_parquet("example.parquet"))

   id  score  label
0   1    0.8      1
1   2    0.6      0
2   3    0.9      1


In [3]:
import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3],
    "score": [0.8, 0.6, 0.9],
    "label": [1, 0, 1]
})

# 쓰기
df.to_parquet("example.parquet", engine="pyarrow", index=False)

# 읽기
df2 = pd.read_parquet("example.parquet", engine="pyarrow")

print(df2)

   id  score  label
0   1    0.8      1
1   2    0.6      0
2   3    0.9      1


In [4]:
#일부 컬럼만 익기
cols = ["id", "score"]
df = pd.read_parquet("example.parquet", columns=cols)

In [6]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("example.parquet")

print("row groups:", pf.num_row_groups)
print("rows:", pf.metadata.num_rows)
print("schema:", pf.schema)

row groups: 1
rows: 3
schema: <pyarrow._parquet.ParquetSchema object at 0x000002106DBDE880>
required group field_id=-1 schema {
  optional int64 field_id=-1 id;
  optional double field_id=-1 score;
  optional int64 field_id=-1 label;
}



In [7]:
import pandas as pd
import numpy as np

n = 1000
rng = np.random.default_rng(42)

df = pd.DataFrame({
    "id": np.arange(1, n + 1, dtype="int64"),
    "age": rng.integers(20, 65, size=n, dtype="int16"),
    "income": rng.normal(0.5, 0.15, size=n).astype("float32"),
    "treatment": rng.integers(0, 2, size=n, dtype="int8"),
})

df["group"] = pd.Series(
    rng.choice(["A", "B"], size=n, p=[0.55, 0.45]),
    dtype="category"
)

# outcome 생성 (처치 효과 약간 포함)
logit = (
    0.03 * df["age"]
    + 0.8 * df["income"]
    + 0.6 * df["treatment"]
)
p = 1 / (1 + np.exp(-logit))
df["outcome"] = (rng.random(n) < p).astype("int8")

df["date"] = pd.to_datetime(
    rng.choice(pd.date_range("2024-01-01", "2024-01-31"), size=n)
)

# Parquet 저장
df.to_parquet(
    "sample_research.parquet",
    engine="pyarrow",
    index=False
)

print("saved: sample_research.parquet")
print(df.head())


saved: sample_research.parquet
   id  age    income  treatment group  outcome       date
0   1   26  0.485861          0     A        0 2024-01-02
1   2   24  0.236341          0     A        1 2024-01-28
2   3   64  0.279943          1     B        1 2024-01-26
3   4   54  0.819387          0     A        1 2024-01-26
4   5   64  0.306887          0     B        1 2024-01-13


In [10]:
import pyarrow
pd.__version__, pyarrow.__version__

('3.0.5', '25.0.0')